# Bloque A.2 y A.3 — Enrollment y Reconocimiento (prototipo con archivo local)

**Proyecto:** Sistema de Reconocimiento Facial para Control de Asistencia (Gimnasio)
**Fase:** CRISP-ML(Q) — Fase 3, Model Engineering

### Objetivo
Construir dos funciones reutilizables:
1. **`enrolar_persona(nombre, fotos)`** — matricula a una persona: extrae el embedding de cada foto que le pases y lo guarda, asociado a su nombre.
2. **`reconocer_persona(foto_nueva)`** — dado un rostro nuevo, lo compara contra **todas** las personas matriculadas y devuelve quién es (o "desconocido" si no coincide con nadie).

Todavía NO usamos PostgreSQL — guardamos las matrículas en un archivo JSON simple junto al notebook. Esto es intencional: nos deja probar la lógica de negocio (enrolar, reconocer, umbral de decisión) de forma aislada, sin la complejidad extra de una base de datos real todavía. El Bloque C solo va a cambiar *dónde* se guardan los embeddings, no *cómo* se comparan — por eso vale la pena dejar bien resuelta la lógica aquí primero.

## Configuración: mismo modelo y detector validados en el Bloque A.1

Reutilizamos exactamente `Facenet512` + `mtcnn`, que ya confirmamos que separan bien "misma persona" de "persona distinta" en tus propias fotos. También traemos directamente del código fuente de DeepFace sus funciones oficiales de **distancia coseno** y **umbral calibrado**, en vez de escribir nuestra propia fórmula o un número fijo a mano — así garantizamos que la decisión de "misma persona / distinta" en A.2/A.3 sea *exactamente* consistente con lo que ya vimos funcionar en A.1.

In [1]:
import os
import json
from deepface import DeepFace
from deepface.modules.verification import find_cosine_distance, find_threshold

MODELO = "Facenet512"
DETECTOR = "mtcnn"
UMBRAL = find_threshold(MODELO, "cosine")  # el mismo 0.30 que vimos en A.1, pero obtenido
                                             # directamente de DeepFace, no escrito a mano.

CARPETA_FOTOS = "fotos_prueba"
RUTA_BASE_DATOS = "base_embeddings.json"

print(f"Modelo: {MODELO} | Detector: {DETECTOR} | Umbral: {UMBRAL}")

26-10-04 15:26:50 - ⚠️ 
 ⚠️ DEPRECATION WARNING:
 Running 'pip install deepface' alone will no longer be sufficient and will
 NOT install a default backend in an upcoming major release.

 Currently, TensorFlow is included by default, but this behavior will be deprecated.
 Please explicitly specify your preferred backend engine when installing:

   -> pip install deepface[tensorflow]
   -> pip install deepface[pytorch]

 Otherwise, you will encounter 'module not found' errors.


Modelo: Facenet512 | Detector: mtcnn | Umbral: 0.3


## Las funciones de almacenamiento (leer y guardar el archivo JSON)

Estas dos funciones son el "disco duro" temporal de nuestra base de datos de personas. Si el archivo todavía no existe (primera vez que corres esto), empezamos con una base vacía.

In [2]:
def cargar_base_datos():
    if os.path.exists(RUTA_BASE_DATOS):
        with open(RUTA_BASE_DATOS, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def guardar_base_datos(base):
    with open(RUTA_BASE_DATOS, "w", encoding="utf-8") as f:
        json.dump(base, f, indent=2)

## Función de Enrollment (A.2)

**Concepto importante:** guardamos **todos** los embeddings de las fotos que le pases a una persona, no solo un promedio. Esto es deliberado: en el Bloque A.1 vimos que una sola foto con ángulo o iluminación distinta puede alejar el embedding; guardar varias referencias por persona (como las 28 fotos reales que va a tomar cada uno de ustedes) le da al sistema más "versiones" contra las cuales comparar, en vez de depender de una sola.

Si la persona ya existía en la base, sus nuevas fotos se **agregan** a las que ya tenía (matriculación incremental), no las reemplazan.

In [3]:
def enrolar_persona(nombre, rutas_fotos):
    """
    Extrae el embedding de cada foto en `rutas_fotos` y lo guarda en la base de datos local
    bajo el nombre `nombre`. Devuelve cuántos embeddings se agregaron exitosamente.
    """
    base = cargar_base_datos()
    base.setdefault(nombre, [])

    agregados = 0
    for ruta in rutas_fotos:
        try:
            resultado = DeepFace.represent(img_path=ruta, model_name=MODELO,
                                            detector_backend=DETECTOR)
            embedding = resultado[0]["embedding"]
            base[nombre].append({"foto": os.path.basename(ruta), "embedding": embedding})
            agregados += 1
        except ValueError as e:
            mensaje_real = str(e.__cause__) if e.__cause__ is not None else str(e)
            print(f"[AVISO] No se pudo procesar {ruta}: {mensaje_real[:120]}")

    guardar_base_datos(base)
    print(f"'{nombre}' matriculado con {agregados} foto(s). "
          f"Total acumulado para esta persona: {len(base[nombre])}")
    return agregados

## Función de Reconocimiento (A.3)

Recibe una foto nueva, extrae su embedding, y lo compara contra **cada embedding guardado de cada persona matriculada** (no solo contra un promedio por persona). Nos quedamos con la coincidencia **más cercana en toda la base**; si esa distancia mínima está por debajo del umbral, reconocemos a esa persona — si no, devolvemos `None` (desconocido).

In [4]:
def reconocer_persona(ruta_foto_nueva, umbral=UMBRAL):
    """
    Compara el rostro de `ruta_foto_nueva` contra toda la base de datos local.
    Devuelve (nombre_reconocido_o_None, distancia_mas_cercana, foto_que_coincidio).
    """
    base = cargar_base_datos()
    if not base:
        print("[AVISO] La base de datos está vacía. Enrola a alguien primero.")
        return None, None, None

    resultado = DeepFace.represent(img_path=ruta_foto_nueva, model_name=MODELO,
                                    detector_backend=DETECTOR)
    embedding_nuevo = resultado[0]["embedding"]

    mejor_nombre, mejor_distancia, mejor_foto = None, float("inf"), None
    for nombre, registros in base.items():
        for registro in registros:
            distancia = find_cosine_distance(embedding_nuevo, registro["embedding"])
            if distancia < mejor_distancia:
                mejor_nombre, mejor_distancia, mejor_foto = nombre, distancia, registro["foto"]

    if mejor_distancia <= umbral:
        return mejor_nombre, mejor_distancia, mejor_foto
    else:
        # Hubo una persona "más parecida" que las demás, pero ni así pasó el umbral:
        # sigue siendo un desconocido, no la persona más cercana.
        return None, mejor_distancia, mejor_foto

## Prueba de extremo a extremo

Usamos tus mismas fotos de `fotos_prueba`, pero de una forma más honesta que en A.1: matriculamos a "Yo" usando **solo `yo_1`**, y dejamos `yo_2` completamente afuera de la matrícula — así la prueba de reconocimiento es sobre una foto que el sistema **nunca vio antes**, que es justo lo que va a pasar de verdad cada vez que alguien llegue al gimnasio (nunca será exactamente la misma foto con la que se matriculó). `otra_persona` tampoco la matriculamos en nadie, para poder probar el rechazo correcto de alguien desconocido.

In [5]:
# Si ya existe una base de datos de una prueba anterior, la reiniciamos para que esta
# prueba sea limpia y predecible.
if os.path.exists(RUTA_BASE_DATOS):
    os.remove(RUTA_BASE_DATOS)

enrolar_persona("Yo", [os.path.join(CARPETA_FOTOS, "yo_1.png")])

'Yo' matriculado con 1 foto(s). Total acumulado para esta persona: 1


1

In [6]:
print("--- Prueba 1: reconocer con una foto DISTINTA de la misma persona matriculada ---")
nombre, distancia, foto_match = reconocer_persona(os.path.join(CARPETA_FOTOS, "yo_2.png"))
if nombre:
    print(f"Reconocido como: {nombre} (distancia={distancia:.4f}, coincidió con {foto_match})")
else:
    print(f"No reconocido (distancia mínima encontrada: {distancia:.4f})")

print()
print("--- Prueba 2: reconocer con una persona que NUNCA fue matriculada ---")
nombre, distancia, foto_match = reconocer_persona(os.path.join(CARPETA_FOTOS, "otra_persona.png"))
if nombre:
    print(f"[ERROR] Se reconoció incorrectamente como: {nombre} (distancia={distancia:.4f})")
else:
    print(f"Correctamente marcado como DESCONOCIDO (distancia mínima encontrada: {distancia:.4f})")

--- Prueba 1: reconocer con una foto DISTINTA de la misma persona matriculada ---
Reconocido como: Yo (distancia=0.1823, coincidió con yo_1.png)

--- Prueba 2: reconocer con una persona que NUNCA fue matriculada ---
Correctamente marcado como DESCONOCIDO (distancia mínima encontrada: 0.5881)


## ¿Cómo interpretar esta prueba?

* **Prueba 1** debería reconocerte como `"Yo"` con una distancia razonablemente baja, aunque `yo_2` nunca estuvo en la matrícula — eso demuestra que el sistema generaliza a fotos nuevas de la misma persona, no que memorizó una foto exacta.
* **Prueba 2** debería devolver `DESCONOCIDO` — eso demuestra que el sistema no "adivina" ni fuerza un reconocimiento cuando la persona realmente no está matriculada.
* Si alguna de las dos falla, antes de cambiar el umbral revisa primero la calidad/ángulo de la foto de prueba (aprendimos esto en A.1: el detector y la foto importan más que ajustar números).

### Siguiente paso: Bloque B (Evaluación) y Bloque C (base de datos real)
Con esto, el núcleo de la lógica de reconocimiento (matricular + reconocer + rechazar desconocidos) queda probado de forma aislada. El Bloque C reemplaza únicamente `cargar_base_datos()` / `guardar_base_datos()` por consultas a PostgreSQL con `pgvector` — `enrolar_persona()` y `reconocer_persona()` casi no van a cambiar.